In [1]:
import os
from pathlib import Path
import plotly.express as px
import plotly.graph_objects as go
# żeby wykresy się tu wyświetlały

import plotly.io as pio
pio.renderers.default="browser"
# jedno pokazuje wykresy w przeglądarce drugie w vscode, json powinno ukrywać całkiem wykresy
#pio.renderers.default="vscode"

# do wybierania defaultowych kolorów w plotly
colors=px.colors.qualitative.Plotly
import pandas as pd



from scipy.stats import mannwhitneyu

In [2]:
main_dir=Path(f"/home/user/studia/licencjat")
known_dir=Path(f"{main_dir}/known")
unknown_dir=Path(f"{main_dir}/unknown")

ic_file_path=f"{main_dir}/GO_information_content_swiss-model.txt"


summary_dir=Path(f"{main_dir}/summary_v3")
os.makedirs(summary_dir, exist_ok=True)

results_dir=Path(f"{main_dir}/summary_v3/results")
os.makedirs(results_dir, exist_ok=True)

figure_dir=Path(f"{main_dir}/summary_v3/figures")
os.makedirs(figure_dir, exist_ok=True)


# Definicje funkcji

In [7]:
def domains_to_list(file_name):
    '''Funkcja zamieniająca csv z kolumną domain_number w listę numerów domen'''
    
    return list(set(pd.read_csv(file_name)["domain_number"]))

In [8]:
def stats_on_DeepFRI_results_separate_proteins(domain_list,known,chainsaw,score_threshold,ic_threshold): 
    """Funkcja zapisująca statystyki o wszystkich go-termach z podziałem na białka (samo MF)"""

    if known==True:
        is_known="known"
        family_dir=known_dir
    else:
        is_known="unknown"
        family_dir=unknown_dir
    if chainsaw==True:
        is_chainsaw="chainsaw"
    else:
        is_chainsaw="pfam"
        
    ic_df=pd.read_csv(ic_file_path, sep=" ", header=None, names=["GO_ID", "IC"]).set_index("GO_ID")
    
    
        
    results_list=[]
    for domain in domain_list:
        try:
            deepfri_results_dir=f"{family_dir}/{domain}_deepfri_{is_chainsaw}"
            mf_results_file=f"{deepfri_results_dir}/{domain}_all_fragments_MF_predictions.csv"
                    
            df=pd.read_csv(mf_results_file)
            df["Protein"]=df["Protein"].str.split("_fragment").str[0]
            df=df.rename(columns={"GO_term/EC":"GO_ID"})
            df=df.merge(ic_df, on="GO_ID", how="left")

            if(ic_threshold!=0):
                df=df[df["IC"]>ic_threshold]
            if(score_threshold!=0):
                df=df[df["Score"]>score_threshold]
                        

            grouped_domain_results=df.groupby(["Protein"])["Score"].agg(median_score="median",count="count").reset_index()
                    
            grouped_domain_results=grouped_domain_results.rename(columns={"Protein":"protein"})
            grouped_domain_results["median_score"]=grouped_domain_results["median_score"].round(4)
            grouped_domain_results["domain_number"]=domain
                
            results_list.append(grouped_domain_results)
            
        
        except:
            print(f"No results for {domain}")

               
    results_df=pd.concat(results_list).reset_index(drop=True)
    results_df=results_df[["domain_number","protein","count","median_score"]]
    
            
    if 'results_df' in locals():
        results_df.to_csv(f"{results_dir}/mf_100_score-{score_threshold}_ic-{ic_threshold}_{is_known}_{is_chainsaw}_separate_proteins.csv", encoding='utf-8', index=False)
        del results_df
    
    
    

In [9]:
def stats_on_DeepFRI_results_ic_median_separate_proteins(domain_list,known,chainsaw): 
    """Funkcja zapisująca medianę IC dla wszystkich domen z listy (osobno dla każdego białka)
    (tylko dla MF i BP - tu samo MF)"""
    
    if known==True:
        is_known="known"
        family_dir=known_dir
    else:
        is_known="unknown"
        family_dir=unknown_dir
    if chainsaw==True:
        is_chainsaw="chainsaw"
    else:
        is_chainsaw="pfam"
        
    ic_df=pd.read_csv(ic_file_path, sep=" ", header=None, names=["GO_ID", "IC"]).set_index("GO_ID")
    
    results_list=[]
    for domain in domain_list:
        try:
            deepfri_results_dir=f"{family_dir}/{domain}_deepfri_{is_chainsaw}"
            mf_results_file=f"{deepfri_results_dir}/{domain}_all_fragments_MF_predictions.csv"
                    
            df=pd.read_csv(mf_results_file)
            df["Protein"]=df["Protein"].str.split("_fragment").str[0]
            df=df.rename(columns={"GO_term/EC":"GO_ID"})
            df=df.merge(ic_df, on="GO_ID", how="left")

            
            df=df.dropna()         

            grouped_domain_results=df.groupby(["Protein"])["IC"].agg(median_ic="median").reset_index()
                    
            grouped_domain_results=grouped_domain_results.rename(columns={"Protein":"protein"})
            grouped_domain_results["median_ic"]=grouped_domain_results["median_ic"].round(4)
            grouped_domain_results["domain_number"]=domain
                
            results_list.append(grouped_domain_results)
            
        
        except:
            print(f"No results for {domain}")

               
    results_df=pd.concat(results_list).reset_index(drop=True)
    results_df=results_df[["domain_number","protein","median_ic"]]
    
            
    if 'results_df' in locals():
        results_df.to_csv(f"{results_dir}/mf_100_{is_known}_{is_chainsaw}_ic_median_separate_families.csv", encoding='utf-8', index=False)
        del results_df
    

In [10]:
def add_stars(col1,col2,col3,p_df):
    p_val=p_df.loc[(p_df["category_id"]==col1)&(p_df["stats"]==col2)&(p_df["data_type"]==col3),"pvalue_bh"].values[0]
    return "***" if p_val<0.001 else "**" if p_val<0.01 else "*" if p_val<0.05 else "no"

In [ ]:
def make_4_violinplots_separate_proteins(known_pfam_results,unknown_pfam_results,known_chainsaw_results,unknown_chainsaw_results,what,plot_title,y_title,results_file,col1,col2,pvalue_df):

    
    fig=go.Figure()

    fig.add_trace(go.Violin(
        y=known_pfam_results[what],
        x=["known"]*len(known_pfam_results),
        name="Pfam",
        box_visible=True,
        marker_color=colors[0],
        customdata=known_pfam_results[["domain_number","protein","count","median_score"]].fillna("NaN").values,
        hovertemplate="<b>%{customdata[0]}<br>(%{customdata[1]})</b><br>count=%{customdata[2]}<br>median_score=%{customdata[3]}<extra></extra>"

    ))
    fig.add_trace(go.Violin(
        y=unknown_pfam_results[what],
        x=["unknown"]*len(unknown_pfam_results),
        name="Pfam",
        box_visible=True,
        marker_color=colors[0],
        customdata=unknown_pfam_results[["domain_number","protein","count","median_score"]].fillna("NaN").values,
        hovertemplate="<b>%{customdata[0]}<br>(%{customdata[1]})</b><br>count=%{customdata[2]}<br>median_score=%{customdata[3]}<extra></extra>",
        showlegend=False

    ))

    fig.add_trace(go.Violin(
        y=known_chainsaw_results[what],
        x=["known"]*len(known_chainsaw_results),
        name="Chainsaw",
        box_visible=True,
        marker_color=colors[1],
        customdata=known_chainsaw_results[["domain_number","protein","count","median_score"]].fillna("NaN").values,
        hovertemplate="<b>%{customdata[0]}<br>(%{customdata[1]})</b><br>count=%{customdata[2]}<br>median_score=%{customdata[3]}<extra></extra>"
        
    ))
    fig.add_trace(go.Violin(
        y=unknown_chainsaw_results[what],
        x=["unknown"]*len(unknown_chainsaw_results),
        name="Chainsaw",
        box_visible=True,
        marker_color=colors[1],
        customdata=unknown_chainsaw_results[["domain_number","protein","count","median_score"]].fillna("NaN").values,
        hovertemplate="<b>%{customdata[0]}<br>(%{customdata[1]})</b><br>count=%{customdata[2]}<br>median_score=%{customdata[3]}<extra></extra>",
        showlegend=False

    ))


    fig.update_layout(
        violinmode="group",
        yaxis_title=y_title,
        title=plot_title,
        font=dict(
            family="Times New Roman",
            size=30
        ),
        #title_font_size=25
    )

    p_known=add_stars(col1,col2,"known",pvalue_df)
    p_unknown=add_stars(col1,col2,"unknown",pvalue_df)

    if p_known!="no":
        if p_known=="*":
            xval=-0.15
        if p_known=="**":
            xval=-0.125
        if p_known=="***":
            xval=-0.1
        y_max=max(
            known_pfam_results[what].quantile(0.75)+1.5*(known_pfam_results[what].quantile(0.75)-known_pfam_results[what].quantile(0.25)),
            known_chainsaw_results[what].quantile(0.75)+1.5*(known_chainsaw_results[what].quantile(0.75)-known_chainsaw_results[what].quantile(0.25))
            )
        y=y_max*1.05

        # dodawanie lini
        fig.add_shape(
            type="line",
            xref="x", yref="y",
            x0=-0.26, x1=0.09,
            y0=y, y1=y,
            line=dict(color="black",width=2)
        )
        
        fig.add_shape(
            type="line",
            xref="x", yref="y",
            x0=-0.26, x1=-0.26,
            y0=y, y1=y-0.02*y,
            line=dict(color="black",width=2)
        )

        fig.add_shape(
            type="line",
            xref="x", yref="y",
            x0=0.09, x1=0.09,
            y0=y, y1=y-0.02*y,
            line=dict(color="black",width=2)
        )

        # dodawainie gwiazdek
        fig.add_annotation(
            x=-0.1,
            y=y*1.01,
            text=p_known,
            showarrow=False,
            xanchor="center"
            
        )


    if p_unknown!="no":
        if p_unknown=="*":
            xval=1.1
        if p_unknown=="**":
            xval=1.0975
        if p_unknown=="***":
            xval=1.095
        y_max=max(
            unknown_pfam_results[what].quantile(0.75)+1.5*(unknown_pfam_results[what].quantile(0.75)-unknown_pfam_results[what].quantile(0.25)),
            unknown_chainsaw_results[what].quantile(0.75)+1.5*(unknown_chainsaw_results[what].quantile(0.75)-unknown_chainsaw_results[what].quantile(0.25))
            )
        y=y_max*1.05
        
        # dodawanie lini
        fig.add_shape(
            type="line",
            xref="x", yref="y",
            x0=0.91, x1=1.26,
            y0=y, y1=y,
            line=dict(color="black",width=2)
        )
        
        fig.add_shape(
            type="line",
            xref="x", yref="y",
            x0=0.91, x1=0.91,
            y0=y, y1=y-0.02*y,
            line=dict(color="black",width=2)
        )
        
        fig.add_shape(
            type="line",
            xref="x", yref="y",
            x0=1.26, x1=1.26,
            y0=y, y1=y-0.02*y,
            line=dict(color="black",width=2)
        )
        
        # dodawanie gwiazdek
        fig.add_annotation(
            x=xval,
            y=y*1.01,
            text=p_unknown,
            showarrow=False,
            xanchor="center"
        )

    fig.show()
    
    plot_path=Path(results_file)
    fig.write_html(plot_path)



In [ ]:
def make_4_violin_separate_proteins_ic_median(known_pfam_results,unknown_pfam_results,known_chainsaw_results,unknown_chainsaw_results,what,plot_title,y_title,results_file,col1,col2,pvalue_df):

    
    fig=go.Figure()

    fig.add_trace(go.Violin(
        y=known_pfam_results[what],
        x=["known"]*len(known_pfam_results),
        name="Pfam",
        box_visible=True,
        marker_color=colors[0],
        customdata=known_pfam_results[["domain_number","protein","median_ic"]].fillna("NaN").values,
        hovertemplate="<b>%{customdata[0]}<br>(%{customdata[1]})</b><br>median_ic=%{customdata[2]}<extra></extra>"

    ))
    fig.add_trace(go.Violin(
        y=unknown_pfam_results[what],
        x=["unknown"]*len(unknown_pfam_results),
        name="Pfam",
        box_visible=True,
        marker_color=colors[0],
        customdata=unknown_pfam_results[["domain_number","protein","median_ic"]].fillna("NaN").values,
        hovertemplate="<b>%{customdata[0]}<br>(%{customdata[1]})</b><br>median_ic=%{customdata[2]}<extra></extra>",
        showlegend=False

    ))

    fig.add_trace(go.Violin(
        y=known_chainsaw_results[what],
        x=["known"]*len(known_chainsaw_results),
        name="Chainsaw",
        box_visible=True,
        marker_color=colors[1],
        customdata=known_chainsaw_results[["domain_number","protein","median_ic"]].fillna("NaN").values,
        hovertemplate="<b>%{customdata[0]}<br>(%{customdata[1]})</b><br>median_ic=%{customdata[2]}<extra></extra>"
        
    ))
    fig.add_trace(go.Violin(
        y=unknown_chainsaw_results[what],
        x=["unknown"]*len(unknown_chainsaw_results),
        name="Chainsaw",
        box_visible=True,
        marker_color=colors[1],
        customdata=unknown_chainsaw_results[["domain_number","protein","median_ic"]].fillna("NaN").values,
        hovertemplate="<b>%{customdata[0]}<br>(%{customdata[1]})</b><br>median_ic=%{customdata[2]}<extra></extra>",
        showlegend=False

    ))


    fig.update_layout(
        violinmode="group",
        yaxis_title=y_title,
        title=plot_title,
        font=dict(
            family="Times New Roman",
            size=20
        ),
        title_font_size=25
    )

    p_known=add_stars(col1,col2,"known",pvalue_df)
    p_unknown=add_stars(col1,col2,"unknown",pvalue_df)

    if p_known!="no":
        if p_known=="*":
            xval=-0.15
        if p_known=="**":
            xval=-0.125
        if p_known=="***":
            xval=-0.1
        y_max=max(
            known_pfam_results[what].quantile(0.75)+1.5*(known_pfam_results[what].quantile(0.75)-known_pfam_results[what].quantile(0.25)),
            known_chainsaw_results[what].quantile(0.75)+1.5*(known_chainsaw_results[what].quantile(0.75)-known_chainsaw_results[what].quantile(0.25))
            )
        y=y_max*1.05

        # dodawanie lini
        fig.add_shape(
            type="line",
            xref="x", yref="y",
            x0=-0.26, x1=0.09,
            y0=y, y1=y,
            line=dict(color="black",width=2)
        )

        fig.add_shape(
            type="line",
            xref="x", yref="y",
            x0=-0.26, x1=-0.26,
            y0=y, y1=y-0.02*y,
            line=dict(color="black",width=2)
        )

        fig.add_shape(
            type="line",
            xref="x", yref="y",
            x0=0.09, x1=0.09,
            y0=y, y1=y-0.02*y,
            line=dict(color="black",width=2)
        )

        # dodawanie gwiazdek
        fig.add_annotation(
            x=-0.1,
            y=y*1.01,
            text=p_known,
            showarrow=False,
            xanchor="center"
            
        )


    if p_unknown!="no":
        if p_unknown=="*":
            xval=1.1
        if p_unknown=="**":
            xval=1.0975
        if p_unknown=="***":
            xval=1.095
        y_max=max(
            unknown_pfam_results[what].quantile(0.75)+1.5*(unknown_pfam_results[what].quantile(0.75)-unknown_pfam_results[what].quantile(0.25)),
            unknown_chainsaw_results[what].quantile(0.75)+1.5*(unknown_chainsaw_results[what].quantile(0.75)-unknown_chainsaw_results[what].quantile(0.25))
            )
        y=y_max*1.05

        # dodawanie lini
        fig.add_shape(
            type="line",
            xref="x", yref="y",
            x0=0.91, x1=1.26,
            y0=y, y1=y,
            line=dict(color="black",width=2)
        )
        
        fig.add_shape(
            type="line",
            xref="x", yref="y",
            x0=0.91, x1=0.91,
            y0=y, y1=y-0.02*y,
            line=dict(color="black",width=2)
        )
        
        fig.add_shape(
            type="line",
            xref="x", yref="y",
            x0=1.26, x1=1.26,
            y0=y, y1=y-0.02*y,
            line=dict(color="black",width=2)
        )
        
        # dodawanie gwiazdek
        fig.add_annotation(
            x=xval,
            y=y*1.01,
            text=p_unknown,
            showarrow=False,
            xanchor="center"
        )

    fig.show()
    
    plot_path=Path(results_file)
    fig.write_html(plot_path)



In [13]:
def test_pfam_chainsaw(known_pfam_results,unknown_pfam_results,known_chainsaw_results,unknown_chainsaw_results,what,what_name,go_type,category,pvalue_df):
   
    stat, p_value=mannwhitneyu(known_pfam_results[what],known_chainsaw_results[what])
    
    pvalue_df.append({
        "category_id":f"{go_type.upper()}_{category}",
        "stats":what_name,
        "data_type":"known",
        "pvalue":p_value
    })

    print(f"Domains with known function:\nStatistics={round(stat,2)}, p-value={round(p_value,2)}")

    alpha=0.05
    if p_value<alpha:
        print("There is a significant difference between results from Pfam and Chainsaw")
    else:
        print("There is no significant difference between results from Pfam and Chainsaw")

    print(f"{'-'*50}")

    stat, p_value=mannwhitneyu(unknown_pfam_results[what],unknown_chainsaw_results[what])

    pvalue_df.append({
        "category_id":f"{go_type.upper()}_{category}",
        "stats":what_name,
        "data_type":"unknown",
        "pvalue":p_value
    })
    
    print(f"Domains of unknown function:\nStatistics={round(stat,2)}, p-value={round(p_value,2)}")

    alpha=0.05
    if p_value<alpha:
        print("There is a significant difference between results from Pfam and Chainsaw")
    else:
        print("There is no significant difference between results from Pfam and Chainsaw")



# Przygotowanie wszystkich wyników (osobno dla wszystkich białek)

In [14]:
# score>0.2

# domeny z Pfam - domeny o znanej funkcji
stats_on_DeepFRI_results_separate_proteins(domains_to_list(f"{main_dir}/accessions_random_100_known_function.csv"),known=True,chainsaw=False,ic_threshold=0,score_threshold=0.2)
# domeny z Chainsaw - domeny o znanej funkcji
stats_on_DeepFRI_results_separate_proteins(domains_to_list(f"{main_dir}/known_100_lengths.csv"),known=True,chainsaw=True,ic_threshold=0,score_threshold=0.2)
# domeny z Pfam - domeny o nieznanej funkcji
stats_on_DeepFRI_results_separate_proteins(domains_to_list(f"{main_dir}/accessions_random_100_unknown_function.csv"),known=False,chainsaw=False,ic_threshold=0,score_threshold=0.2)
# domeny z Chainsaw - domeny o nieznanej funkcji
stats_on_DeepFRI_results_separate_proteins(domains_to_list(f"{main_dir}/unknown_100_lengths.csv"),known=False,chainsaw=True,ic_threshold=0,score_threshold=0.2)


No results for Knl1_RWD_C


In [15]:
# score>0.3

# domeny z Pfam - domeny o znanej funkcji
stats_on_DeepFRI_results_separate_proteins(domains_to_list(f"{main_dir}/accessions_random_100_known_function.csv"),known=True,chainsaw=False,ic_threshold=0,score_threshold=0.3)
# domeny z Chainsaw - domeny o znanej funkcji
stats_on_DeepFRI_results_separate_proteins(domains_to_list(f"{main_dir}/known_100_lengths.csv"),known=True,chainsaw=True,ic_threshold=0,score_threshold=0.3)
# domeny z Pfam - domeny o nieznanej funkcji
stats_on_DeepFRI_results_separate_proteins(domains_to_list(f"{main_dir}/accessions_random_100_unknown_function.csv"),known=False,chainsaw=False,ic_threshold=0,score_threshold=0.3)
# domeny z Chainsaw - domeny o nieznanej funkcji
stats_on_DeepFRI_results_separate_proteins(domains_to_list(f"{main_dir}/unknown_100_lengths.csv"),known=False,chainsaw=True,ic_threshold=0,score_threshold=0.3)


No results for Knl1_RWD_C


In [16]:
# score>0.5

# domeny z Pfam - domeny o znanej funkcji
stats_on_DeepFRI_results_separate_proteins(domains_to_list(f"{main_dir}/accessions_random_100_known_function.csv"),known=True,chainsaw=False,ic_threshold=0,score_threshold=0.5)
# domeny z Chainsaw - domeny o znanej funkcji
stats_on_DeepFRI_results_separate_proteins(domains_to_list(f"{main_dir}/known_100_lengths.csv"),known=True,chainsaw=True,ic_threshold=0,score_threshold=0.5)
# domeny z Pfam - domeny o nieznanej funkcji
stats_on_DeepFRI_results_separate_proteins(domains_to_list(f"{main_dir}/accessions_random_100_unknown_function.csv"),known=False,chainsaw=False,ic_threshold=0,score_threshold=0.5)
# domeny z Chainsaw - domeny o nieznanej funkcji
stats_on_DeepFRI_results_separate_proteins(domains_to_list(f"{main_dir}/unknown_100_lengths.csv"),known=False,chainsaw=True,ic_threshold=0,score_threshold=0.5)


No results for Knl1_RWD_C


In [17]:
# score>0.2 && IC>5

# domeny z Pfam - domeny o znanej funkcji
stats_on_DeepFRI_results_separate_proteins(domains_to_list(f"{main_dir}/accessions_random_100_known_function.csv"),known=True,chainsaw=False,ic_threshold=5,score_threshold=0.2)
# domeny z Chainsaw - domeny o znanej funkcji
stats_on_DeepFRI_results_separate_proteins(domains_to_list(f"{main_dir}/known_100_lengths.csv"),known=True,chainsaw=True,ic_threshold=5,score_threshold=0.2)
# domeny z Pfam - domeny o nieznanej funkcji
stats_on_DeepFRI_results_separate_proteins(domains_to_list(f"{main_dir}/accessions_random_100_unknown_function.csv"),known=False,chainsaw=False,ic_threshold=5,score_threshold=0.2)
# domeny z Chainsaw - domeny o nieznanej funkcji
stats_on_DeepFRI_results_separate_proteins(domains_to_list(f"{main_dir}/unknown_100_lengths.csv"),known=False,chainsaw=True,ic_threshold=5,score_threshold=0.2)


No results for Knl1_RWD_C


In [18]:
# score>0.3 && IC>5

# domeny z Pfam - domeny o znanej funkcji
stats_on_DeepFRI_results_separate_proteins(domains_to_list(f"{main_dir}/accessions_random_100_known_function.csv"),known=True,chainsaw=False,ic_threshold=5,score_threshold=0.3)
# domeny z Chainsaw - domeny o znanej funkcji
stats_on_DeepFRI_results_separate_proteins(domains_to_list(f"{main_dir}/known_100_lengths.csv"),known=True,chainsaw=True,ic_threshold=5,score_threshold=0.3)
# domeny z Pfam - domeny o nieznanej funkcji
stats_on_DeepFRI_results_separate_proteins(domains_to_list(f"{main_dir}/accessions_random_100_unknown_function.csv"),known=False,chainsaw=False,ic_threshold=5,score_threshold=0.3)
# domeny z Chainsaw - domeny o nieznanej funkcji
stats_on_DeepFRI_results_separate_proteins(domains_to_list(f"{main_dir}/unknown_100_lengths.csv"),known=False,chainsaw=True,ic_threshold=5,score_threshold=0.3)


No results for Knl1_RWD_C


In [19]:
# score>0.5 && IC>5

# domeny z Pfam - domeny o znanej funkcji
stats_on_DeepFRI_results_separate_proteins(domains_to_list(f"{main_dir}/accessions_random_100_known_function.csv"),known=True,chainsaw=False,ic_threshold=5,score_threshold=0.5)
# domeny z Chainsaw - domeny o znanej funkcji
stats_on_DeepFRI_results_separate_proteins(domains_to_list(f"{main_dir}/known_100_lengths.csv"),known=True,chainsaw=True,ic_threshold=5,score_threshold=0.5)
# domeny z Pfam - domeny o nieznanej funkcji
stats_on_DeepFRI_results_separate_proteins(domains_to_list(f"{main_dir}/accessions_random_100_unknown_function.csv"),known=False,chainsaw=False,ic_threshold=5,score_threshold=0.5)
# domeny z Chainsaw - domeny o nieznanej funkcji
stats_on_DeepFRI_results_separate_proteins(domains_to_list(f"{main_dir}/unknown_100_lengths.csv"),known=False,chainsaw=True,ic_threshold=5,score_threshold=0.5)


No results for Knl1_RWD_C


# Liczenie p-value
Na początek, żeby można było je dodać do wykresów

In [20]:
pvalue_df=[]

In [21]:
# zakresy domen

known_results=f"{main_dir}/known_100_lengths.csv"
try:
    len_comparison_known=pd.read_csv(known_results)
except FileNotFoundError:
    print(f"File {known_results} doesn't exist")
    

unknown_results=f"{main_dir}/unknown_100_lengths.csv"
try:
    len_comparison_unknown=pd.read_csv(unknown_results)
except FileNotFoundError:
    print(f"File {unknown_results} doesn't exist")



In [22]:
# porównanie percent overlap z chainsaw jako bazą
stat, p_value=mannwhitneyu(len_comparison_known["percent_overlap_chainsaw"],len_comparison_unknown["percent_overlap_chainsaw"])

pvalue_df.append({
    "category_id":f"lengths",
    "stats":"percent_overlap_chainsaw",
    "data_type":"difference",
    "pvalue":p_value
})

print(f"Statistics={round(stat,2)}, p-value={round(p_value,2)}")

alpha=0.05
if p_value<alpha:
    print("There is a significant difference between results from Pfam and Chainsaw")
else:
    print("There is no significant difference between results from Pfam and Chainsaw")


Statistics=4519857.0, p-value=0.0
There is a significant difference between results from Pfam and Chainsaw


In [23]:
# porównanie długości

stat, p_value=mannwhitneyu(len_comparison_known["len_pfam"],len_comparison_known["len_chainsaw"])
    
pvalue_df.append({
    "category_id":f"lengths",
    "stats":"length",
    "data_type":"known",
    "pvalue":p_value
})
    
print(f"Domains with known function:\nStatistics={round(stat,2)}, p-value={round(p_value,2)}")

alpha=0.05
if p_value<alpha:
    print("There is a significant difference between results from Pfam and Chainsaw")
else:
    print("There is no significant difference between results from Pfam and Chainsaw")

print(f"{'-'*50}")

stat, p_value=mannwhitneyu(len_comparison_unknown["len_pfam"],len_comparison_unknown["len_chainsaw"])
    
pvalue_df.append({
    "category_id":f"lengths",
    "stats":"length",
    "data_type":"unknown",
    "pvalue":p_value
})
    
print(f"Domains of unknown function:\nStatistics={round(stat,2)}, p-value={round(p_value,2)}")

alpha=0.05
if p_value<alpha:
    print("There is a significant difference between results from Pfam and Chainsaw")
else:
    print("There is no significant difference between results from Pfam and Chainsaw")



Domains with known function:
Statistics=10502273.0, p-value=0.0
There is a significant difference between results from Pfam and Chainsaw
--------------------------------------------------
Domains of unknown function:
Statistics=4674859.5, p-value=0.02
There is a significant difference between results from Pfam and Chainsaw


In [24]:
known_pfam_results=pd.read_csv(f"{results_dir}/mf_100_score-0.2_ic-0_known_pfam_separate_proteins.csv")
unknown_pfam_results=pd.read_csv(f"{results_dir}/mf_100_score-0.2_ic-0_unknown_pfam_separate_proteins.csv")
known_chainsaw_results=pd.read_csv(f"{results_dir}/mf_100_score-0.2_ic-0_known_chainsaw_separate_proteins.csv")
unknown_chainsaw_results=pd.read_csv(f"{results_dir}/mf_100_score-0.2_ic-0_unknown_chainsaw_separate_proteins.csv")

test_pfam_chainsaw(
    known_pfam_results,
    unknown_pfam_results,
    known_chainsaw_results,
    unknown_chainsaw_results,
    "count",
    "count",
    "MF",
    "02",
    pvalue_df
)

Domains with known function:
Statistics=7311380.0, p-value=0.76
There is no significant difference between results from Pfam and Chainsaw
--------------------------------------------------
Domains of unknown function:
Statistics=1730006.5, p-value=0.4
There is no significant difference between results from Pfam and Chainsaw


In [25]:
known_pfam_results=pd.read_csv(f"{results_dir}/mf_100_score-0.3_ic-0_known_pfam_separate_proteins.csv")
unknown_pfam_results=pd.read_csv(f"{results_dir}/mf_100_score-0.3_ic-0_unknown_pfam_separate_proteins.csv")
known_chainsaw_results=pd.read_csv(f"{results_dir}/mf_100_score-0.3_ic-0_known_chainsaw_separate_proteins.csv")
unknown_chainsaw_results=pd.read_csv(f"{results_dir}/mf_100_score-0.3_ic-0_unknown_chainsaw_separate_proteins.csv")

test_pfam_chainsaw(
    known_pfam_results,
    unknown_pfam_results,
    known_chainsaw_results,
    unknown_chainsaw_results,
    "count",
    "count",
    "MF",
    "03",
    pvalue_df
)

Domains with known function:
Statistics=4047122.0, p-value=0.22
There is no significant difference between results from Pfam and Chainsaw
--------------------------------------------------
Domains of unknown function:
Statistics=820393.0, p-value=0.12
There is no significant difference between results from Pfam and Chainsaw


In [26]:
known_pfam_results=pd.read_csv(f"{results_dir}/mf_100_score-0.5_ic-0_known_pfam_separate_proteins.csv")
unknown_pfam_results=pd.read_csv(f"{results_dir}/mf_100_score-0.5_ic-0_unknown_pfam_separate_proteins.csv")
known_chainsaw_results=pd.read_csv(f"{results_dir}/mf_100_score-0.5_ic-0_known_chainsaw_separate_proteins.csv")
unknown_chainsaw_results=pd.read_csv(f"{results_dir}/mf_100_score-0.5_ic-0_unknown_chainsaw_separate_proteins.csv")

test_pfam_chainsaw(
    known_pfam_results,
    unknown_pfam_results,
    known_chainsaw_results,
    unknown_chainsaw_results,
    "count",
    "count",
    "MF",
    "05",
    pvalue_df
)

Domains with known function:
Statistics=1461555.5, p-value=0.23
There is no significant difference between results from Pfam and Chainsaw
--------------------------------------------------
Domains of unknown function:
Statistics=200410.0, p-value=0.0
There is a significant difference between results from Pfam and Chainsaw


In [27]:
known_pfam_results=pd.read_csv(f"{results_dir}/mf_100_score-0.2_ic-5_known_pfam_separate_proteins.csv")
unknown_pfam_results=pd.read_csv(f"{results_dir}/mf_100_score-0.2_ic-5_unknown_pfam_separate_proteins.csv")
known_chainsaw_results=pd.read_csv(f"{results_dir}/mf_100_score-0.2_ic-5_known_chainsaw_separate_proteins.csv")
unknown_chainsaw_results=pd.read_csv(f"{results_dir}/mf_100_score-0.2_ic-5_unknown_chainsaw_separate_proteins.csv")

test_pfam_chainsaw(
    known_pfam_results,
    unknown_pfam_results,
    known_chainsaw_results,
    unknown_chainsaw_results,
    "count",
    "count",
    "MF",
    "ic_02",
    pvalue_df
)

Domains with known function:
Statistics=2181554.0, p-value=0.37
There is no significant difference between results from Pfam and Chainsaw
--------------------------------------------------
Domains of unknown function:
Statistics=551333.5, p-value=0.0
There is a significant difference between results from Pfam and Chainsaw


In [28]:
known_pfam_results=pd.read_csv(f"{results_dir}/mf_100_score-0.3_ic-5_known_pfam_separate_proteins.csv")
unknown_pfam_results=pd.read_csv(f"{results_dir}/mf_100_score-0.3_ic-5_unknown_pfam_separate_proteins.csv")
known_chainsaw_results=pd.read_csv(f"{results_dir}/mf_100_score-0.3_ic-5_known_chainsaw_separate_proteins.csv")
unknown_chainsaw_results=pd.read_csv(f"{results_dir}/mf_100_score-0.3_ic-5_unknown_chainsaw_separate_proteins.csv")

test_pfam_chainsaw(
    known_pfam_results,
    unknown_pfam_results,
    known_chainsaw_results,
    unknown_chainsaw_results,
    "count",
    "count",
    "MF",
    "ic_03",
    pvalue_df
)

Domains with known function:
Statistics=1010616.0, p-value=0.03
There is a significant difference between results from Pfam and Chainsaw
--------------------------------------------------
Domains of unknown function:
Statistics=227773.5, p-value=0.04
There is a significant difference between results from Pfam and Chainsaw


In [29]:
known_pfam_results=pd.read_csv(f"{results_dir}/mf_100_score-0.5_ic-5_known_pfam_separate_proteins.csv")
unknown_pfam_results=pd.read_csv(f"{results_dir}/mf_100_score-0.5_ic-5_unknown_pfam_separate_proteins.csv")
known_chainsaw_results=pd.read_csv(f"{results_dir}/mf_100_score-0.5_ic-5_known_chainsaw_separate_proteins.csv")
unknown_chainsaw_results=pd.read_csv(f"{results_dir}/mf_100_score-0.5_ic-5_unknown_chainsaw_separate_proteins.csv")

test_pfam_chainsaw(
    known_pfam_results,
    unknown_pfam_results,
    known_chainsaw_results,
    unknown_chainsaw_results,
    "count",
    "count",
    "MF",
    "ic_05",
    pvalue_df
)

Domains with known function:
Statistics=308439.0, p-value=0.13
There is no significant difference between results from Pfam and Chainsaw
--------------------------------------------------
Domains of unknown function:
Statistics=46979.0, p-value=0.0
There is a significant difference between results from Pfam and Chainsaw


In [30]:
known_pfam_results=pd.read_csv(f"{results_dir}/mf_100_score-0.2_ic-0_known_pfam_separate_proteins.csv")
unknown_pfam_results=pd.read_csv(f"{results_dir}/mf_100_score-0.2_ic-0_unknown_pfam_separate_proteins.csv")
known_chainsaw_results=pd.read_csv(f"{results_dir}/mf_100_score-0.2_ic-0_known_chainsaw_separate_proteins.csv")
unknown_chainsaw_results=pd.read_csv(f"{results_dir}/mf_100_score-0.2_ic-0_unknown_chainsaw_separate_proteins.csv")

test_pfam_chainsaw(
    known_pfam_results,
    unknown_pfam_results,
    known_chainsaw_results,
    unknown_chainsaw_results,
    "median_score",
    "median_score",
    "MF",
    "02",
    pvalue_df
)

Domains with known function:
Statistics=6954660.5, p-value=0.0
There is a significant difference between results from Pfam and Chainsaw
--------------------------------------------------
Domains of unknown function:
Statistics=1673678.5, p-value=0.01
There is a significant difference between results from Pfam and Chainsaw


In [31]:
known_pfam_results=pd.read_csv(f"{results_dir}/mf_100_score-0.3_ic-0_known_pfam_separate_proteins.csv")
unknown_pfam_results=pd.read_csv(f"{results_dir}/mf_100_score-0.3_ic-0_unknown_pfam_separate_proteins.csv")
known_chainsaw_results=pd.read_csv(f"{results_dir}/mf_100_score-0.3_ic-0_known_chainsaw_separate_proteins.csv")
unknown_chainsaw_results=pd.read_csv(f"{results_dir}/mf_100_score-0.3_ic-0_unknown_chainsaw_separate_proteins.csv")

test_pfam_chainsaw(
    known_pfam_results,
    unknown_pfam_results,
    known_chainsaw_results,
    unknown_chainsaw_results,
    "median_score",
    "median_score",
    "MF",
    "03",
    pvalue_df
)

Domains with known function:
Statistics=3833014.5, p-value=0.02
There is a significant difference between results from Pfam and Chainsaw
--------------------------------------------------
Domains of unknown function:
Statistics=813164.0, p-value=0.06
There is no significant difference between results from Pfam and Chainsaw


In [32]:
known_pfam_results=pd.read_csv(f"{results_dir}/mf_100_score-0.5_ic-0_known_pfam_separate_proteins.csv")
unknown_pfam_results=pd.read_csv(f"{results_dir}/mf_100_score-0.5_ic-0_unknown_pfam_separate_proteins.csv")
known_chainsaw_results=pd.read_csv(f"{results_dir}/mf_100_score-0.5_ic-0_known_chainsaw_separate_proteins.csv")
unknown_chainsaw_results=pd.read_csv(f"{results_dir}/mf_100_score-0.5_ic-0_unknown_chainsaw_separate_proteins.csv")

test_pfam_chainsaw(
    known_pfam_results,
    unknown_pfam_results,
    known_chainsaw_results,
    unknown_chainsaw_results,
    "median_score",
    "median_score",
    "MF",
    "05",
    pvalue_df
)

Domains with known function:
Statistics=1406927.5, p-value=0.45
There is no significant difference between results from Pfam and Chainsaw
--------------------------------------------------
Domains of unknown function:
Statistics=216528.0, p-value=0.37
There is no significant difference between results from Pfam and Chainsaw


In [33]:
known_pfam_results=pd.read_csv(f"{results_dir}/mf_100_score-0.2_ic-5_known_pfam_separate_proteins.csv")
unknown_pfam_results=pd.read_csv(f"{results_dir}/mf_100_score-0.2_ic-5_unknown_pfam_separate_proteins.csv")
known_chainsaw_results=pd.read_csv(f"{results_dir}/mf_100_score-0.2_ic-5_known_chainsaw_separate_proteins.csv")
unknown_chainsaw_results=pd.read_csv(f"{results_dir}/mf_100_score-0.2_ic-5_unknown_chainsaw_separate_proteins.csv")

test_pfam_chainsaw(
    known_pfam_results,
    unknown_pfam_results,
    known_chainsaw_results,
    unknown_chainsaw_results,
    "median_score",
    "median_score",
    "MF",
    "ic_02",
    pvalue_df
)

Domains with known function:
Statistics=2000599.5, p-value=0.0
There is a significant difference between results from Pfam and Chainsaw
--------------------------------------------------
Domains of unknown function:
Statistics=535872.0, p-value=0.0
There is a significant difference between results from Pfam and Chainsaw


In [34]:
known_pfam_results=pd.read_csv(f"{results_dir}/mf_100_score-0.3_ic-5_known_pfam_separate_proteins.csv")
unknown_pfam_results=pd.read_csv(f"{results_dir}/mf_100_score-0.3_ic-5_unknown_pfam_separate_proteins.csv")
known_chainsaw_results=pd.read_csv(f"{results_dir}/mf_100_score-0.3_ic-5_known_chainsaw_separate_proteins.csv")
unknown_chainsaw_results=pd.read_csv(f"{results_dir}/mf_100_score-0.3_ic-5_unknown_chainsaw_separate_proteins.csv")

test_pfam_chainsaw(
    known_pfam_results,
    unknown_pfam_results,
    known_chainsaw_results,
    unknown_chainsaw_results,
    "median_score",
    "median_score",
    "MF",
    "ic_03",
    pvalue_df
)

Domains with known function:
Statistics=901190.5, p-value=0.0
There is a significant difference between results from Pfam and Chainsaw
--------------------------------------------------
Domains of unknown function:
Statistics=229615.0, p-value=0.09
There is no significant difference between results from Pfam and Chainsaw


In [35]:
known_pfam_results=pd.read_csv(f"{results_dir}/mf_100_score-0.5_ic-5_known_pfam_separate_proteins.csv")
unknown_pfam_results=pd.read_csv(f"{results_dir}/mf_100_score-0.5_ic-5_unknown_pfam_separate_proteins.csv")
known_chainsaw_results=pd.read_csv(f"{results_dir}/mf_100_score-0.5_ic-5_known_chainsaw_separate_proteins.csv")
unknown_chainsaw_results=pd.read_csv(f"{results_dir}/mf_100_score-0.5_ic-5_unknown_chainsaw_separate_proteins.csv")

test_pfam_chainsaw(
    known_pfam_results,
    unknown_pfam_results,
    known_chainsaw_results,
    unknown_chainsaw_results,
    "median_score",
    "median_score",
    "MF",
    "ic_05",
    pvalue_df
)

Domains with known function:
Statistics=275938.0, p-value=0.02
There is a significant difference between results from Pfam and Chainsaw
--------------------------------------------------
Domains of unknown function:
Statistics=57174.0, p-value=0.38
There is no significant difference between results from Pfam and Chainsaw


In [36]:
known_pfam_results=pd.read_csv(f"{results_dir}/mf_100_known_pfam_ic_median_separate_families.csv")
known_chainsaw_results=pd.read_csv(f"{results_dir}/mf_100_known_chainsaw_ic_median_separate_families.csv")
unknown_pfam_results=pd.read_csv(f"{results_dir}/mf_100_unknown_pfam_ic_median_separate_families.csv")
unknown_chainsaw_results=pd.read_csv(f"{results_dir}/mf_100_unknown_chainsaw_ic_median_separate_families.csv")

test_pfam_chainsaw(
    known_pfam_results,
    unknown_pfam_results,
    known_chainsaw_results,
    unknown_chainsaw_results,
    "median_ic",
    "median_ic_comparison",
    "MF",
    'median_ic',
    pvalue_df
)

Domains with known function:
Statistics=9759958.5, p-value=0.0
There is a significant difference between results from Pfam and Chainsaw
--------------------------------------------------
Domains of unknown function:
Statistics=2962240.5, p-value=0.0
There is a significant difference between results from Pfam and Chainsaw


In [37]:
from statsmodels.stats.multitest import multipletests
pvalue_df=pd.DataFrame(pvalue_df)
pvalue_df["pvalue_bh"]=multipletests(pvalue_df["pvalue"], method="fdr_bh")[1]

pvalue_df.to_csv(f"{summary_dir}/pvalue.csv",index=False)



# Wyniki

## Chainsaw vs Pfam - porównania zakresów domen

In [38]:
# ładowanie obu plików


known_results=f"{main_dir}/known_100_lengths.csv"
try:
    len_comparison_known=pd.read_csv(known_results)
except FileNotFoundError:
    print(f"File {known_results} doesn't exist")
    
    

unknown_results=f"{main_dir}/unknown_100_lengths.csv"
try:
    len_comparison_unknown=pd.read_csv(unknown_results)
except FileNotFoundError:
    print(f"File {unknown_results} doesn't exist")

In [43]:


fig=go.Figure()

fig.add_trace(go.Violin(
    y=len_comparison_known["percent_overlap_chainsaw"],
    x=["known"]*len(len_comparison_known),
    name="known",
    #boxpoints="all",
    box_visible=True,
    marker_color=colors[2],
    customdata=len_comparison_known["domain_number"],
    hovertemplate="Percent overlap=%{y}<br>Domain=%{customdata}<extra></extra>",
    showlegend=False
    
))
fig.add_trace(go.Violin(
    y=len_comparison_unknown["percent_overlap_chainsaw"],
    x=["unknown"]*len(len_comparison_unknown),
    name="unknown",
    #boxpoints="all",
    box_visible=True,
    marker_color=colors[2],
    customdata=len_comparison_unknown["domain_number"],
    hovertemplate="Percent overlap=%{y}<br>Domain=%{customdata}<extra></extra>",
    showlegend=False
    
))

fig.update_layout(
    #violinmode="group", 
    yaxis_title="Percent overlap",
    title="Percent overlap (compared to Chainsaw) of 100 known and unknown domains",
    font=dict(
        family="Times New Roman",
        size=20
    ),
    title_font_size=25
)

p=add_stars("lengths","percent_overlap_chainsaw","difference",pvalue_df)

if p!="no":

    y_max=max(len_comparison_known["percent_overlap_chainsaw"].max(),len_comparison_unknown["percent_overlap_chainsaw"].max())
    y=y_max*1.05
    
    # dodawanie lini
    fig.add_shape(
        type="line",
        xref="x", yref="y",
        x0=0, x1=1,
        y0=y, y1=y,
        line=dict(color="black",width=2)
    )

    fig.add_shape(
        type="line",
        xref="x", yref="y",
        x0=0, x1=0,
        y0=y, y1=y-0.02*y,
        line=dict(color="black",width=2)
    )

    fig.add_shape(
        type="line",
        xref="x", yref="y",
        x0=1, x1=1,
        y0=y, y1=y-0.02*y,
        line=dict(color="black",width=2)
    )

    # dodawanie gwiazdek
    fig.add_annotation(
        x=0.5,
        y=y*1.01,
        text=p,
        showarrow=False,
        xanchor="center"

    )

fig.show()


plot_path=Path(f"{figure_dir}/known_unknown_100_chainsaw_percent_overlap_plot.html")
fig.write_html(plot_path)

In [52]:

fig=go.Figure()

fig.add_trace(go.Violin(
    y=len_comparison_known["len_pfam"],
    x=["known"]*len(len_comparison_known),
    name="Pfam",
    #boxpoints="all",
    box_visible=True,
    marker_color=colors[0],
    customdata=len_comparison_known["domain_number"],
    hovertemplate="Length=%{y}<br>Domain=%{customdata}<extra></extra>"
    
))
fig.add_trace(go.Violin(
    y=len_comparison_unknown["len_pfam"],
    x=["unknown"]*len(len_comparison_unknown),
    name="Pfam",
    #boxpoints="all",
    box_visible=True,
    marker_color=colors[0],
    customdata=len_comparison_unknown["domain_number"],
    hovertemplate="Length=%{y}<br>Domain=%{customdata}<extra></extra>",
    showlegend=False
    
))

fig.add_trace(go.Violin(
    y=len_comparison_known["len_chainsaw"],
    x=["known"]*len(len_comparison_known),
    name="Chainsaw",
    #boxpoints="all",
    box_visible=True,
    marker_color=colors[1],
    customdata=len_comparison_known["domain_number"],
    hovertemplate="Length=%{y}<br>Domain=%{customdata}<extra></extra>"
    
))
fig.add_trace(go.Violin(
    y=len_comparison_unknown["len_chainsaw"],
    x=["unknown"]*len(len_comparison_unknown),
    name="Chainsaw",
    #boxpoints="all",
    box_visible=True,
    marker_color=colors[1],
    customdata=len_comparison_unknown["domain_number"],
    hovertemplate="Length=%{y}<br>Domain=%{customdata}<extra></extra>",
    showlegend=False
    
))

fig.update_layout(
    violinmode="group",
    #boxgap=0.1,
    #boxgroupgap=0.05,
    yaxis_title="Length",
    title="Length comparison of 100 known and unknown domains",
    font=dict(
        family="Times New Roman",
        size=20
    ),
    title_font_size=25
)

p_known=add_stars("lengths","length","known",pvalue_df)
p_unknown=add_stars("lengths","length","unknown",pvalue_df)

if p_known!="no":
    if p_known=="*":
        xval=-0.15
    if p_known=="**":
        xval=-0.125
    if p_known=="***":
        xval=-0.1
    y_max=max(
        len_comparison_known["len_pfam"].quantile(0.75)+1.5*(len_comparison_known["len_pfam"].quantile(0.75)-len_comparison_known["len_pfam"].quantile(0.25)),
        len_comparison_known["len_chainsaw"].quantile(0.75)+1.5*(len_comparison_known["len_chainsaw"].quantile(0.75)-len_comparison_known["len_chainsaw"].quantile(0.25))
        )
    y=y_max*1.05

    # dodawanie lini
    fig.add_shape(
        type="line",
        xref="x", yref="y",
        x0=-0.26, x1=0.09,
        y0=y, y1=y,
        line=dict(color="black",width=2)
    )

    fig.add_shape(
        type="line",
        xref="x", yref="y",
        x0=-0.26, x1=-0.26,
        y0=y, y1=y-0.02*y,
        line=dict(color="black",width=2)
    )

    fig.add_shape(
        type="line",
        xref="x", yref="y",
        x0=0.09, x1=0.09,
        y0=y, y1=y-0.02*y,
        line=dict(color="black",width=2)
    )

    # dodawanie gwiazdek
    fig.add_annotation(
        x=-0.1,
        y=y*1.01,
        text=p_known,
        showarrow=False,
        xanchor="center"
        
    )


if p_unknown!="no":
    if p_unknown=="*":
        xval=1.1
    if p_unknown=="**":
        xval=1.0975
    if p_unknown=="***":
        xval=1.095
    y_max=max(
        len_comparison_unknown["len_pfam"].quantile(0.75)+1.5*(len_comparison_unknown["len_pfam"].quantile(0.75)-len_comparison_unknown["len_pfam"].quantile(0.25)),
        len_comparison_unknown["len_chainsaw"].quantile(0.75)+1.5*(len_comparison_unknown["len_chainsaw"].quantile(0.75)-len_comparison_unknown["len_chainsaw"].quantile(0.25))
        )
    y=y_max*1.05
    
    # dodawanie lini
    fig.add_shape(
        type="line",
        xref="x", yref="y",
        x0=0.91, x1=1.26,
        y0=y, y1=y,
        line=dict(color="black",width=2)
    )
    
    fig.add_shape(
        type="line",
        xref="x", yref="y",
        x0=0.91, x1=0.91,
        y0=y, y1=y-0.02*y,
        line=dict(color="black",width=2)
    )

    fig.add_shape(
        type="line",
        xref="x", yref="y",
        x0=1.26, x1=1.26,
        y0=y, y1=y-0.02*y,
        line=dict(color="black",width=2)
    )
    
    # dodawanie gwiazdek
    fig.add_annotation(
        x=xval,
        y=y*1.01,
        text=p_unknown,
        showarrow=False,
        xanchor="center"
    )

fig.show()





plot_path=Path(f"{figure_dir}/known_unknown_100_lengths_plot.html")
fig.write_html(plot_path)

## Porównania wyników MF z DeepFRI

### Ilości wyników


In [37]:
known_pfam_results=pd.read_csv(f"{results_dir}/mf_100_score-0.2_ic-0_known_pfam_separate_proteins.csv")
unknown_pfam_results=pd.read_csv(f"{results_dir}/mf_100_score-0.2_ic-0_unknown_pfam_separate_proteins.csv")
known_chainsaw_results=pd.read_csv(f"{results_dir}/mf_100_score-0.2_ic-0_known_chainsaw_separate_proteins.csv")
unknown_chainsaw_results=pd.read_csv(f"{results_dir}/mf_100_score-0.2_ic-0_unknown_chainsaw_separate_proteins.csv")


make_4_boxplots_separate_proteins(
    known_pfam_results,
    unknown_pfam_results,
    known_chainsaw_results,
    unknown_chainsaw_results,
    "count",
    f"score>0.2",
    "Count",
    f"{figure_dir}/mf_100_count_comparison_02.html",
    "MF_02",
    "count",
    pvalue_df
)






In [38]:
known_pfam_results=pd.read_csv(f"{results_dir}/mf_100_score-0.3_ic-0_known_pfam_separate_proteins.csv")
unknown_pfam_results=pd.read_csv(f"{results_dir}/mf_100_score-0.3_ic-0_unknown_pfam_separate_proteins.csv")
known_chainsaw_results=pd.read_csv(f"{results_dir}/mf_100_score-0.3_ic-0_known_chainsaw_separate_proteins.csv")
unknown_chainsaw_results=pd.read_csv(f"{results_dir}/mf_100_score-0.3_ic-0_unknown_chainsaw_separate_proteins.csv")

make_4_boxplots_separate_proteins(
    known_pfam_results,
    unknown_pfam_results,
    known_chainsaw_results,
    unknown_chainsaw_results,
    "count",
    f"score>0.3",
    "Count",
    f"{figure_dir}/mf_100_count_comparison_03.html",
    "MF_03",
    "count",
    pvalue_df
)


In [39]:
known_pfam_results=pd.read_csv(f"{results_dir}/mf_100_score-0.5_ic-0_known_pfam_separate_proteins.csv")
unknown_pfam_results=pd.read_csv(f"{results_dir}/mf_100_score-0.5_ic-0_unknown_pfam_separate_proteins.csv")
known_chainsaw_results=pd.read_csv(f"{results_dir}/mf_100_score-0.5_ic-0_known_chainsaw_separate_proteins.csv")
unknown_chainsaw_results=pd.read_csv(f"{results_dir}/mf_100_score-0.5_ic-0_unknown_chainsaw_separate_proteins.csv")

make_4_boxplots_separate_proteins(
    known_pfam_results,
    unknown_pfam_results,
    known_chainsaw_results,
    unknown_chainsaw_results,
    "count",
    f"score>0.5",
    "Count",
    f"{figure_dir}/mf_100_count_comparison_05.html",
    "MF_05",
    "count",
    pvalue_df
)


In [40]:
known_pfam_results=pd.read_csv(f"{results_dir}/mf_100_score-0.2_ic-5_known_pfam_separate_proteins.csv")
unknown_pfam_results=pd.read_csv(f"{results_dir}/mf_100_score-0.2_ic-5_unknown_pfam_separate_proteins.csv")
known_chainsaw_results=pd.read_csv(f"{results_dir}/mf_100_score-0.2_ic-5_known_chainsaw_separate_proteins.csv")
unknown_chainsaw_results=pd.read_csv(f"{results_dir}/mf_100_score-0.2_ic-5_unknown_chainsaw_separate_proteins.csv")


make_4_boxplots_separate_proteins(
    known_pfam_results,
    unknown_pfam_results,
    known_chainsaw_results,
    unknown_chainsaw_results,
    "count",
    f"score>0.2 && IC>5",
    "Count",
    f"{figure_dir}/mf_100_count_comparison_ic_02.html",
    "MF_ic_02",
    "count",
    pvalue_df
)



In [41]:
known_pfam_results=pd.read_csv(f"{results_dir}/mf_100_score-0.3_ic-5_known_pfam_separate_proteins.csv")
unknown_pfam_results=pd.read_csv(f"{results_dir}/mf_100_score-0.3_ic-5_unknown_pfam_separate_proteins.csv")
known_chainsaw_results=pd.read_csv(f"{results_dir}/mf_100_score-0.3_ic-5_known_chainsaw_separate_proteins.csv")
unknown_chainsaw_results=pd.read_csv(f"{results_dir}/mf_100_score-0.3_ic-5_unknown_chainsaw_separate_proteins.csv")

make_4_boxplots_separate_proteins(
    known_pfam_results,
    unknown_pfam_results,
    known_chainsaw_results,
    unknown_chainsaw_results,
    "count",
    f"score>0.3 && IC>5",
    "Count",
    f"{figure_dir}/mf_100_count_comparison_ic_03.html",
    "MF_ic_03",
    "count",
    pvalue_df
)


In [42]:
known_pfam_results=pd.read_csv(f"{results_dir}/mf_100_score-0.5_ic-5_known_pfam_separate_proteins.csv")
unknown_pfam_results=pd.read_csv(f"{results_dir}/mf_100_score-0.5_ic-5_unknown_pfam_separate_proteins.csv")
known_chainsaw_results=pd.read_csv(f"{results_dir}/mf_100_score-0.5_ic-5_known_chainsaw_separate_proteins.csv")
unknown_chainsaw_results=pd.read_csv(f"{results_dir}/mf_100_score-0.5_ic-5_unknown_chainsaw_separate_proteins.csv")


make_4_boxplots_separate_proteins(
    known_pfam_results,
    unknown_pfam_results,
    known_chainsaw_results,
    unknown_chainsaw_results,
    "count",
    f"score>0.5 && IC>5",
    "Count",
    f"{figure_dir}/mf_100_count_comparison_ic_05.html",
    "MF_ic_05",
    "count",
    pvalue_df
)


### Jakości wyników

In [43]:
known_pfam_results=pd.read_csv(f"{results_dir}/mf_100_score-0.2_ic-0_known_pfam_separate_proteins.csv")
unknown_pfam_results=pd.read_csv(f"{results_dir}/mf_100_score-0.2_ic-0_unknown_pfam_separate_proteins.csv")
known_chainsaw_results=pd.read_csv(f"{results_dir}/mf_100_score-0.2_ic-0_known_chainsaw_separate_proteins.csv")
unknown_chainsaw_results=pd.read_csv(f"{results_dir}/mf_100_score-0.2_ic-0_unknown_chainsaw_separate_proteins.csv")

make_4_boxplots_separate_proteins(
    known_pfam_results,
    unknown_pfam_results,
    known_chainsaw_results,
    unknown_chainsaw_results,
    "median_score",
    f"score>0.2",
    "Median score",
    f"{figure_dir}/mf_100_median_score_comparison_02.html",
    "MF_02",
    "median_score",
    pvalue_df
)



In [44]:
known_pfam_results=pd.read_csv(f"{results_dir}/mf_100_score-0.3_ic-0_known_pfam_separate_proteins.csv")
unknown_pfam_results=pd.read_csv(f"{results_dir}/mf_100_score-0.3_ic-0_unknown_pfam_separate_proteins.csv")
known_chainsaw_results=pd.read_csv(f"{results_dir}/mf_100_score-0.3_ic-0_known_chainsaw_separate_proteins.csv")
unknown_chainsaw_results=pd.read_csv(f"{results_dir}/mf_100_score-0.3_ic-0_unknown_chainsaw_separate_proteins.csv")


make_4_boxplots_separate_proteins(
    known_pfam_results,
    unknown_pfam_results,
    known_chainsaw_results,
    unknown_chainsaw_results,
    "median_score",
    f"score>0.3",
    "Median score",
    f"{figure_dir}/mf_100_median_score_comparison_03.html",
    "MF_03",
    "median_score",
    pvalue_df
)


In [45]:
known_pfam_results=pd.read_csv(f"{results_dir}/mf_100_score-0.5_ic-0_known_pfam_separate_proteins.csv")
unknown_pfam_results=pd.read_csv(f"{results_dir}/mf_100_score-0.5_ic-0_unknown_pfam_separate_proteins.csv")
known_chainsaw_results=pd.read_csv(f"{results_dir}/mf_100_score-0.5_ic-0_known_chainsaw_separate_proteins.csv")
unknown_chainsaw_results=pd.read_csv(f"{results_dir}/mf_100_score-0.5_ic-0_unknown_chainsaw_separate_proteins.csv")

make_4_boxplots_separate_proteins(
    known_pfam_results,
    unknown_pfam_results,
    known_chainsaw_results,
    unknown_chainsaw_results,
    "median_score",
    f"score>0.5",
    "Median score",
    f"{figure_dir}/mf_100_median_score_comparison_05.html",
    "MF_05",
    "median_score",
    pvalue_df
)


In [46]:
known_pfam_results=pd.read_csv(f"{results_dir}/mf_100_score-0.2_ic-5_known_pfam_separate_proteins.csv")
unknown_pfam_results=pd.read_csv(f"{results_dir}/mf_100_score-0.2_ic-5_unknown_pfam_separate_proteins.csv")
known_chainsaw_results=pd.read_csv(f"{results_dir}/mf_100_score-0.2_ic-5_known_chainsaw_separate_proteins.csv")
unknown_chainsaw_results=pd.read_csv(f"{results_dir}/mf_100_score-0.2_ic-5_unknown_chainsaw_separate_proteins.csv")


make_4_boxplots_separate_proteins(
    known_pfam_results,
    unknown_pfam_results,
    known_chainsaw_results,
    unknown_chainsaw_results,
    "median_score",
    f"score>0.2 && IC>5",
    "Median score",
    f"{figure_dir}/mf_100_median_score_comparison_ic_02.html",
    "MF_ic_02",
    "median_score",
    pvalue_df
)



In [47]:
known_pfam_results=pd.read_csv(f"{results_dir}/mf_100_score-0.3_ic-5_known_pfam_separate_proteins.csv")
unknown_pfam_results=pd.read_csv(f"{results_dir}/mf_100_score-0.3_ic-5_unknown_pfam_separate_proteins.csv")
known_chainsaw_results=pd.read_csv(f"{results_dir}/mf_100_score-0.3_ic-5_known_chainsaw_separate_proteins.csv")
unknown_chainsaw_results=pd.read_csv(f"{results_dir}/mf_100_score-0.3_ic-5_unknown_chainsaw_separate_proteins.csv")


make_4_boxplots_separate_proteins(
    known_pfam_results,
    unknown_pfam_results,
    known_chainsaw_results,
    unknown_chainsaw_results,
    "median_score",
    f"score>0.3 && IC>5",
    "Median score",
    f"{figure_dir}/mf_100_median_score_comparison_ic_03.html",
    "MF_ic_03",
    "median_score",
    pvalue_df
)



In [48]:
known_pfam_results=pd.read_csv(f"{results_dir}/mf_100_score-0.5_ic-5_known_pfam_separate_proteins.csv")
unknown_pfam_results=pd.read_csv(f"{results_dir}/mf_100_score-0.5_ic-5_unknown_pfam_separate_proteins.csv")
known_chainsaw_results=pd.read_csv(f"{results_dir}/mf_100_score-0.5_ic-5_known_chainsaw_separate_proteins.csv")
unknown_chainsaw_results=pd.read_csv(f"{results_dir}/mf_100_score-0.5_ic-5_unknown_chainsaw_separate_proteins.csv")


make_4_boxplots_separate_proteins(
    known_pfam_results,
    unknown_pfam_results,
    known_chainsaw_results,
    unknown_chainsaw_results,
    "median_score",
    f"score>0.5 && IC>5",
    "Median score",
    f"{figure_dir}/mf_100_median_score_comparison_ic_05.html",
    "MF_ic_05",
    "median_score",
    pvalue_df
)



### Mediany IC

In [49]:

# domeny z Pfam - domeny o znanej funkcji
stats_on_DeepFRI_results_ic_median_separate_proteins(domains_to_list(f"{main_dir}/accessions_random_100_known_function.csv"),known=True,chainsaw=False)
# domeny z Chainsaw - domeny o znanej funkcji
stats_on_DeepFRI_results_ic_median_separate_proteins(domains_to_list(f"{main_dir}/known_100_lengths.csv"),known=True,chainsaw=True)
# domeny z Pfam - domeny o nieznanej funkcji
stats_on_DeepFRI_results_ic_median_separate_proteins(domains_to_list(f"{main_dir}/accessions_random_100_unknown_function.csv"),known=False,chainsaw=False)
# domeny z Chainsaw - domeny o nieznanej funkcji
stats_on_DeepFRI_results_ic_median_separate_proteins(domains_to_list(f"{main_dir}/unknown_100_lengths.csv"),known=False,chainsaw=True)

No results for Knl1_RWD_C


In [50]:
known_pfam_results=pd.read_csv(f"{results_dir}/mf_100_known_pfam_ic_median_separate_families.csv")
known_chainsaw_results=pd.read_csv(f"{results_dir}/mf_100_known_chainsaw_ic_median_separate_families.csv")
unknown_pfam_results=pd.read_csv(f"{results_dir}/mf_100_unknown_pfam_ic_median_separate_families.csv")
unknown_chainsaw_results=pd.read_csv(f"{results_dir}/mf_100_unknown_chainsaw_ic_median_separate_families.csv")


In [ ]:

make_4_violinplots_separate_proteins_ic_median(
    known_pfam_results,
    unknown_pfam_results,
    known_chainsaw_results,
    unknown_chainsaw_results,
    "median_ic",
    f"MF IC median comparison of 100 known and unknown domains",
    "IC median",
    f"{figure_dir}/mf_100_median_separate_proteins.html",
    "MF_median_ic",
    "median_ic_comparison",
    pvalue_df
)

# Sprawdzenie ile jest krótkich domen wśród wszystkich białek

ilość_białek_z_domeną_z_pfam_>50/ilość_białek_z_pfam

(to jest zrobione tylko dla białek w których chainsaw znalazł domenę zgodną z założeniami)

In [52]:
# odsetek krótkich domen z pfam o znanej funkcji

lengths_results=Path(f"{main_dir}/known_100_lengths.csv")
          
df_lengths=pd.read_csv(lengths_results)
len_all1=len(df_lengths)
            

df_lengths=df_lengths[df_lengths["len_pfam"]<=50]# d_lengths ma teraz te fragmenty które były krótsze niż 50, poniżej są one usuwane
df_lengths=df_lengths.copy()
len_short1=len(df_lengths)

print(len_short1/len_all1)

0.1108636052090473


In [53]:
# odsetek krótkich domen z pfam o nieznanej funkcji
          
lengths_results=Path(f"{main_dir}/unknown_100_lengths.csv")
            
df_lengths=pd.read_csv(lengths_results)
len_all2=len(df_lengths)
            

df_lengths=df_lengths[df_lengths["len_pfam"]<=50]# d_lengths ma teraz te fragmenty które były krótsze niż 50, poniżej są one usuwane
df_lengths=df_lengths.copy()
len_short2=len(df_lengths)

print(len_short2/len_all2)

0.04885888781742205


In [54]:
print((len_short1+len_short2)/(len_all1+len_all2))

0.08930367721023807
